# Notebook 8 — Experiments 11 to 15: Subgroup Evaluation and Fairness Gap
**Abhishek Tomar | PN1196973 | LJMU | April 2026**

---

| Exp | Subgroup | Fills |
|---|---|---|
| 11 | Emoji-heavy | Table 4 |
| 12 | Slang-heavy | Table 4 |
| 13 | Sarcasm-indicated | Table 4 |
| 14 | Formal (reference) | Table 4 |
| 15 | Fairness gap comparison | Table 4, Table 5 |

**No training happens in this notebook.** Every experiment reads the saved
predictions from Experiment 10 and partitions them by subgroup. Total runtime is
under twenty minutes.

The sarcasm subgroup has only 14 posts in the TweetEval test partition, which
is small enough that point estimates are not defensible on their own. Bootstrap
confidence intervals are computed for every subgroup so that the uncertainty is
visible rather than implied.

## Cell 1: Setup

In [1]:
!pip install -q emoji xgboost scikit-learn pandas pyarrow

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append(r"/content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation")
from thesis_utils import *

import pandas as pd, numpy as np, scipy.sparse as sp, pickle, time, json
from sklearn.model_selection import GridSearchCV

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 7.5 MB/s eta 0:00:00
Mounted at /content/drive
thesis_utils loaded. [V2 FIXED: Fairlearn EOD, Theil index]
  seed = 42
  subgroup thresholds: emoji > 0.05, slang > 0.1, formal min words = 8
  reference subgroup  = formal


## Cell 2: Load Final Model Predictions

Everything here derives from one saved prediction file.

In [2]:
D = PATHS["data"]

with open(D / "final_model_config.json") as f:
    cfg = json.load(f)

print("Final model configuration from Exp 10:")
for k, v in cfg.items():
    print(f"  {k:<14}: {v}")

pred = load_predictions("exp10", "FinalModel")

print(f"\nPredictions loaded: {len(pred):,} test instances")
print(f"Overall accuracy   : {pred['correct'].mean():.4f}")
print("\nSubgroup sizes in the test partition:")
for sg, n in pred["subgroup"].value_counts().items():
    flag = "   <-- small, bootstrap CI essential" if n < 150 else ""
    print(f"  {sg:<14}: {n:>6,}{flag}")

Final model configuration from Exp 10:
  model         : LogisticRegression
  feature_set   : Hybrid feature set
  feature_exp   : exp09
  best_params   : {'C': '10', 'penalty': 'l2', 'solver': 'liblinear'}
  macro_f1      : 0.5877578201451183
  brier         : 0.1822364883995802

Predictions loaded: 12,284 test instances
Overall accuracy   : 0.5981

Subgroup sizes in the test partition:
  formal        : 10,398
  other         :  1,116
  emoji-heavy   :    696
  slang-heavy   :     60   <-- small, bootstrap CI essential
  sarcasm       :     14   <-- small, bootstrap CI essential


## Cell 3: Experiments 11 to 14 — Per-Subgroup Evaluation

Each experiment filters the saved predictions to one subgroup and computes the
standard metric block plus a bootstrap confidence interval on macro F1.

Experiment 14 (formal) is the reference group against which the other three are
compared in Experiment 15.

In [3]:
SUBGROUP_EXPERIMENTS = [
    ("exp11", "emoji-heavy", "Emoji-Heavy Subgroup Evaluation"),
    ("exp12", "slang-heavy", "Slang-Heavy Subgroup Evaluation"),
    ("exp13", "sarcasm",     "Sarcasm-Indicated Subgroup Evaluation"),
    ("exp14", "formal",      "Formal Text Subgroup Evaluation"),
]

subgroup_rows = []

for exp_id, sg, title in SUBGROUP_EXPERIMENTS:
    part = pred[pred["subgroup"] == sg]
    print("="*62)
    print(f"{exp_id.upper()} — {title}")
    print("="*62)

    if len(part) < 10:
        print(f"  Only {len(part)} instances. Too few to evaluate.\n")
        continue

    yt, yp = part["y_true"].values, part["y_pred"].values
    res = evaluate_model(yt, yp, label=sg)

    mean_f1, lo, hi = bootstrap_ci(yt, yp, n_boot=1000)

    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(yt, yp)
    with np.errstate(divide="ignore", invalid="ignore"):
        fp = cm.sum(axis=0) - np.diag(cm)
        fn = cm.sum(axis=1) - np.diag(cm)
        tp = np.diag(cm)
        tn = cm.sum() - (fp + fn + tp)
        fpr = float(np.nanmean(fp / (fp + tn)))
        fnr = float(np.nanmean(fn / (fn + tp)))

    row = {
        "Subgroup":               sg,
        "Number of Samples":      len(part),
        "Accuracy":               res["Accuracy"],
        "Macro F1":               res["Macro F1"],
        "Macro F1 CI Low":        lo,
        "Macro F1 CI High":       hi,
        "FPR":                    fpr,
        "FNR":                    fnr,
        "Misclassification Rate": 1 - res["Accuracy"],
    }
    subgroup_rows.append(row)

    print(f"  n            : {len(part):,}")
    print(f"  Accuracy     : {res['Accuracy']:.4f}")
    print(f"  Macro F1     : {res['Macro F1']:.4f}   95% CI [{lo:.4f}, {hi:.4f}]")
    print(f"  CI width     : {hi-lo:.4f}"
          f"{'   <-- wide, small sample' if (hi-lo) > 0.15 else ''}")
    print(f"  FPR / FNR    : {fpr:.4f} / {fnr:.4f}")
    print()

subgroup_df = pd.DataFrame(subgroup_rows)

EXP11 — Emoji-Heavy Subgroup Evaluation
  n            : 696
  Accuracy     : 0.6207
  Macro F1     : 0.6022   95% CI [0.5624, 0.6401]
  CI width     : 0.0777
  FPR / FNR    : 0.1990 / 0.4072

EXP12 — Slang-Heavy Subgroup Evaluation
  n            : 60
  Accuracy     : 0.6167
  Macro F1     : 0.6180   95% CI [0.4915, 0.7327]
  CI width     : 0.2412   <-- wide, small sample
  FPR / FNR    : 0.1906 / 0.3627

EXP13 — Sarcasm-Indicated Subgroup Evaluation
  n            : 14
  Accuracy     : 0.7857
  Macro F1     : 0.7897   95% CI [0.4675, 1.0000]
  CI width     : 0.5325   <-- wide, small sample
  FPR / FNR    : 0.1124 / 0.1619

EXP14 — Formal Text Subgroup Evaluation
  n            : 10,398
  Accuracy     : 0.5920
  Macro F1     : 0.5801   95% CI [0.5707, 0.5900]
  CI width     : 0.0194
  FPR / FNR    : 0.2206 / 0.4032



## Cell 4: Experiment 15 — Linguistic Fairness Gap

Compares each informal subgroup against the formal reference group.

The gap columns are deliberately simple and interpretable. The full AIF360 and
Fairlearn audit runs separately in Notebook 12, where the two toolkits are
cross-checked against each other.

In [4]:
formal = subgroup_df[subgroup_df["Subgroup"] == "formal"]
if formal.empty:
    raise ValueError("Formal reference subgroup missing. Cannot compute gaps.")
formal = formal.iloc[0]

print("="*62)
print("EXP 15 — LINGUISTIC FAIRNESS GAP")
print("="*62)
print(f"Reference group: formal  (n={int(formal['Number of Samples']):,}, "
      f"Macro F1={formal['Macro F1']:.4f})\n")

gap_rows = []
for _, r in subgroup_df.iterrows():
    if r["Subgroup"] == "formal":
        continue
    gap = {
        "Subgroup":                 r["Subgroup"],
        "Number of Samples":        int(r["Number of Samples"]),
        "F1 Gap":                   formal["Macro F1"] - r["Macro F1"],
        "FPR Gap":                  r["FPR"] - formal["FPR"],
        "FNR Gap":                  r["FNR"] - formal["FNR"],
        "Error Rate Gap":           r["Misclassification Rate"] - formal["Misclassification Rate"],
    }
    f1g = gap["F1 Gap"]
    gap["Risk Level"] = "High" if f1g > 0.10 else ("Medium" if f1g > 0.05 else "Low")
    gap_rows.append(gap)

    print(f"{r['Subgroup']:<14} vs formal")
    print(f"  F1 Gap         : {gap['F1 Gap']:+.4f}   ({gap['Risk Level']} risk)")
    print(f"  FPR Gap        : {gap['FPR Gap']:+.4f}")
    print(f"  FNR Gap        : {gap['FNR Gap']:+.4f}")
    print(f"  Error Rate Gap : {gap['Error Rate Gap']:+.4f}")
    print()

gap_df = pd.DataFrame(gap_rows)

EXP 15 — LINGUISTIC FAIRNESS GAP
Reference group: formal  (n=10,398, Macro F1=0.5801)

emoji-heavy    vs formal
  F1 Gap         : -0.0221   (Low risk)
  FPR Gap        : -0.0217
  FNR Gap        : +0.0040
  Error Rate Gap : -0.0287

slang-heavy    vs formal
  F1 Gap         : -0.0378   (Low risk)
  FPR Gap        : -0.0300
  FNR Gap        : -0.0405
  Error Rate Gap : -0.0246

sarcasm        vs formal
  F1 Gap         : -0.2096   (Low risk)
  FPR Gap        : -0.1082
  FNR Gap        : -0.2413
  Error Rate Gap : -0.1937



## Cell 5: Table 4 — Subgroup Performance and Fairness Gap

In [5]:
table4 = subgroup_df.merge(
    gap_df[["Subgroup", "F1 Gap", "FPR Gap", "FNR Gap",
            "Error Rate Gap", "Risk Level"]],
    on="Subgroup", how="left")

table4.loc[table4["Subgroup"] == "formal",
           ["F1 Gap", "FPR Gap", "FNR Gap", "Error Rate Gap"]] = 0.0
table4.loc[table4["Subgroup"] == "formal", "Risk Level"] = "Reference"

def interpret(r):
    if r["Subgroup"] == "formal":
        return "Reference group. All comparisons are made against this."
    g = r["F1 Gap"]
    if g > 0.10:
        return (f"Macro F1 is {g:.3f} below formal text. Substantial disparity. "
                f"Sample size {int(r['Number of Samples']):,}.")
    if g > 0.05:
        return f"Moderate disparity of {g:.3f} against formal text."
    if g > 0:
        return f"Small disparity of {g:.3f}. Within acceptable range."
    return "Performs at or above the formal reference group."

table4["Main Interpretation"] = table4.apply(interpret, axis=1)

table4 = table4[["Subgroup", "Number of Samples", "Accuracy", "Macro F1",
                 "Macro F1 CI Low", "Macro F1 CI High", "FPR", "FNR",
                 "Misclassification Rate", "F1 Gap", "FPR Gap", "FNR Gap",
                 "Error Rate Gap", "Risk Level", "Main Interpretation"]]

print("="*100)
print("TABLE 4 — LINGUISTIC SUBGROUP PERFORMANCE AND FAIRNESS GAP")
print("="*100)
print(table4[["Subgroup", "Number of Samples", "Accuracy", "Macro F1",
              "F1 Gap", "Risk Level"]].to_string(index=False))

save_result_table(table4, "Table4_Subgroup_Fairness_Gap")

TABLE 4 — LINGUISTIC SUBGROUP PERFORMANCE AND FAIRNESS GAP
   Subgroup  Number of Samples  Accuracy  Macro F1    F1 Gap Risk Level
emoji-heavy                696  0.620690  0.602209 -0.022077        Low
slang-heavy                 60  0.616667  0.617980 -0.037848        Low
    sarcasm                 14  0.785714  0.789744 -0.209611        Low
     formal              10398  0.592037  0.580132  0.000000  Reference
  saved table -> Table4_Subgroup_Fairness_Gap.csv


PosixPath('/content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation/results/Table4_Subgroup_Fairness_Gap.csv')

## Cell 6: Table 5 — Misclassification Pattern Analysis

Counts every confusion direction per subgroup. This is the quantitative structure
that the methodology feedback asked for: not just a heatmap, but rates and
proportions per subgroup and per error direction.

The taxonomy columns are filled later in Notebook 15, once SHAP and Anchor
evidence is available for each instance.

In [6]:
errors = pred[pred["correct"] == 0]
total_errors = len(errors)

print(f"Total misclassifications: {total_errors:,} "
      f"({total_errors/len(pred)*100:.1f}% of test set)\n")

pattern_rows = []
for sg in pred["subgroup"].unique():
    if sg == "other":
        continue
    sg_all = pred[pred["subgroup"] == sg]
    sg_err = errors[errors["subgroup"] == sg]
    if len(sg_all) == 0:
        continue
    for actual in sorted(sg_err["y_true"].unique()):
        for predicted in sorted(sg_err["y_pred"].unique()):
            if actual == predicted:
                continue
            n = len(sg_err[(sg_err["y_true"] == actual) &
                           (sg_err["y_pred"] == predicted)])
            if n == 0:
                continue
            pattern_rows.append({
                "Error Type":                 f"{actual.capitalize()} to {predicted.capitalize()}",
                "Subgroup Mainly Affected":   sg,
                "Actual Sentiment":           actual,
                "Predicted Sentiment":        predicted,
                "Number of Cases":            n,
                "Percentage of Total Errors": round(n / total_errors * 100, 2),
                "Rate Within Subgroup":       round(n / len(sg_all) * 100, 2),
                "Example Text":               "",
                "Possible Reason":            "",
                "Taxonomy Type":              "",
            })

table5 = pd.DataFrame(pattern_rows).sort_values(
    "Number of Cases", ascending=False).reset_index(drop=True)

print("="*90)
print("TABLE 5 — MISCLASSIFICATION PATTERN ANALYSIS")
print("="*90)
print(table5.head(15)[["Error Type", "Subgroup Mainly Affected",
                       "Number of Cases", "Percentage of Total Errors",
                       "Rate Within Subgroup"]].to_string(index=False))

save_result_table(table5, "Table5_Misclassification_Patterns")

print("\nThe 'Rate Within Subgroup' column matters more than raw counts.")
print("Formal posts produce the most errors in absolute terms simply because")
print("they are 95% of the data. The rate normalises for that.")

Total misclassifications: 4,937 (40.2% of test set)

TABLE 5 — MISCLASSIFICATION PATTERN ANALYSIS
          Error Type Subgroup Mainly Affected  Number of Cases  Percentage of Total Errors  Rate Within Subgroup
 Negative to Neutral                   formal             1491                       30.20                 14.34
 Neutral to Positive                   formal              948                       19.20                  9.12
 Neutral to Negative                   formal              855                       17.32                  8.22
 Positive to Neutral                   formal              518                       10.49                  4.98
Negative to Positive                   formal              341                        6.91                  3.28
Positive to Negative                   formal               89                        1.80                  0.86
 Positive to Neutral              emoji-heavy               89                        1.80                 12.7

## Cell 7: Save Misclassified Instances for Later Notebooks

Notebook 15 needs these for LIME, Anchor and the error taxonomy. Saving them here
means the selection is made once and is identical across every later analysis.

In [7]:
tw_test = pd.read_parquet(D / "tw_test.parquet").reset_index(drop=True)

errors_full = errors.copy()
errors_full["text"]       = tw_test.loc[errors.index, "text"].values
errors_full["text_clean"] = tw_test.loc[errors.index, "text_clean"].values

errors_full = errors_full.sort_values("confidence", ascending=False)
errors_full.to_parquet(D / "misclassified_instances.parquet", index=False)

print(f"Saved {len(errors_full):,} misclassified instances with their text.")
print("\nMost confident errors (these are the most damaging in deployment):")
for _, r in errors_full.head(5).iterrows():
    print(f"  [{r['subgroup']:<12}] {r['y_true']} -> {r['y_pred']}  "
          f"conf={r['confidence']:.3f}")
    print(f"      {str(r['text'])[:95]}")

correct_full = pred[pred["correct"] == 1].copy()
correct_full["text"] = tw_test.loc[correct_full.index, "text"].values
correct_full.to_parquet(D / "correct_instances.parquet", index=False)

print(f"\nAlso saved {len(correct_full):,} correctly classified instances")
print("for the LIME comparison in Notebook 15.")
print("\nNext: Notebook 9 — Experiments 16 and 17")

Saved 4,937 misclassified instances with their text.

Most confident errors (these are the most damaging in deployment):
  [other       ] neutral -> positive  conf=0.998
      @user perfect @user
  [other       ] neutral -> positive  conf=0.997
      Happy @user oppression day
  [formal      ] neutral -> positive  conf=0.994
      @user When will Melania do her "I have a dream" speech? I'm looking forward to it :)
  [formal      ] positive -> neutral  conf=0.994
      The Reputation Doctor weighs in on Tony Romo #NFL @user joins @user on #TheMorningRush LISTEN:
  [other       ] neutral -> positive  conf=0.994
      @user @user @user good mans

Also saved 7,347 correctly classified instances
for the LIME comparison in Notebook 15.

Next: Notebook 9 — Experiments 16 and 17
